In [1]:
import pandas as pd
import numpy as np


In [ ]:
# Build movie embeddings for the full movies.csv catalog using the same feature steps as notebook 03
ratings = pd.read_csv("../data/ml-32m/ratings.csv")
movies = pd.read_csv("../data/ml-32m/movies.csv")
links = pd.read_csv("../data/ml-32m/links.csv")

print("Ratings:", ratings.shape)
print("Movies:", movies.shape)
print("Links:", links.shape)

# genre hot encoding
genre_encoded = movies["genres"].str.get_dummies(sep="|")
movies_encoded = pd.concat([movies, genre_encoded], axis=1)
movies_encoded = movies_encoded.drop(columns=["genres", "(no genres listed)"])

#clean title and extract release year
movies_encoded["Release_Year"] = movies_encoded["title"].str.extract(r"\((\d{4})\)$")[0]
movies_encoded["Release_Year"] = pd.to_numeric(movies_encoded["Release_Year"], errors="coerce")
movies_encoded["title_clean"] = movies_encoded["title"].str.replace(r"\s*\(\d{4}\)$", "", regex=True)
movies_encoded = movies_encoded.drop(columns=["title"])

# movie avg
avg_ratings = ratings.groupby("movieId")["rating"].mean().rename("avg_rating")

#decade encoding
movies_encoded = movies_encoded.merge(avg_ratings, on="movieId", how="left")
movies_encoded["Decade"] = (movies_encoded["Release_Year"] // 10) * 10
# only keep decades represented in training data
valid_decades = range(1890, 2030, 10)
movies_encoded.loc[~movies_encoded["Decade"].isin(valid_decades), "Decade"] = np.nan
movies_encoded = pd.get_dummies(movies_encoded, columns=["Decade"], dtype=int)

tmdb_data = np.load("../data/tmdb_movie_features.npz")
tmdb_df = pd.DataFrame(
    tmdb_data["features"],
    columns=[f"tmdb_{i}" for i in range(tmdb_data["features"].shape[1])]
)
tmdb_df["movieId"] = tmdb_data["movie_ids"]

movie_embeddings = movies_encoded.merge(tmdb_df, on="movieId", how="left")
movie_embeddings["avg_rating"] = movie_embeddings["avg_rating"].fillna(ratings["rating"].mean())
tmdb_cols = [col for col in movie_embeddings.columns if col.startswith("tmdb_")]
movie_embeddings[tmdb_cols] = movie_embeddings[tmdb_cols].fillna(0)

base_movie_cols = [
    col for col in movies_encoded.columns if col not in ["movieId", "title_clean", "Release_Year"]
]
movie_feature_frame = movie_embeddings[base_movie_cols + tmdb_cols].copy()
movie_feature_frame = movie_feature_frame.rename(
    columns={col: f"movie_{col}" for col in movie_feature_frame.columns}
)
movie_feature_cols = [f"movie_{col}" for col in base_movie_cols] + [
    f"movie_{col}" for col in tmdb_cols
]

movie_feature_frame.to_parquet("../data/movie_embeddings.parquet", index=False)
np.save("../data/movie_embedding_cols.npy", movie_feature_cols)

print("Movie embedding shape:", movie_feature_frame.shape)
print(movie_feature_frame.head())
print("Movie feature columns:", movie_feature_cols)

Ratings: (32000204, 4)
Movies: (87585, 3)
Links: (87585, 3)
Movie embedding frame: (87585, 595)
   movie_Action  movie_Adventure  movie_Animation  movie_Children  \
0             0                1                1               1   
1             0                1                0               1   
2             0                0                0               0   
3             0                0                0               0   
4             0                0                0               0   

   movie_Comedy  movie_Crime  movie_Documentary  movie_Drama  movie_Fantasy  \
0             1            0                  0            0              1   
1             0            0                  0            0              1   
2             1            0                  0            0              0   
3             1            0                  0            1              0   
4             1            0                  0            0              0   

   movie_Film-

In [11]:
from tensorflow.keras.models import load_model

movieNN = load_model('../models/movieNN.keras')

movie_embeddings = movieNN.predict(movie_feature_frame)

movie_embeddings_df = pd.DataFrame(movie_embeddings, index=movie_feature_frame.index)

movie_embeddings_df.to_parquet("../data/movie_embeddings_nn.parquet", index=False)

2738/2738 ━━━━━━━━━━━━━━━━━━━━ 1s 430us/step
